# Aerobic Fermentation — Media Stoichiometry from CHNOSP Balance

Derives all stoichiometric coefficients from an elemental (CHNOSP) balance and
tracks how every medium component changes as glucose is consumed.

Two biomass elemental compositions are compared throughout:

| Name | Formula | Source |
|---|---|---|
| Roels extended | CH₁.₈O₀.₅N₀.₂S₀.₀₀₅P₀.₀₁ | Roels (1983), extended with S and P |
| Upcraft | CH₁.₉₈O₀.₆₄₇N₀.₁₇₃P₀.₀₁₅₁ | Upcraft's composition |

**Shared assumptions**
- Carbon + energy source: glucose (C₆H₁₂O₆)
- Nitrogen source: ammonium (NH₄⁺, oxidation state −3 — same as in biomass, so absent from the degree-of-reduction balance)
- Sulfur source: sulfate (SO₄²⁻) — relevant only for Roels extended
- Phosphorus source: dihydrogen phosphate (H₂PO₄⁻)
- Aerobic: O₂ is the terminal electron acceptor
- Yield of biomass on glucose: **Y_{X/S} = 0.4 g g⁻¹** (both compositions)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 1. Biomass compositions

Express biomass per **C-mol** (one formula unit per carbon atom):

$$\text{Biomass} = \text{CH}_{a}\text{O}_{b}\text{N}_{c}\text{S}_{d}\text{P}_{e}$$

The two compositions differ mainly in H/C, O/C and N/C; Upcraft's formula contains no sulfur.

In [ ]:
MW = dict(C=12.011, H=1.008, O=15.999, N=14.007, S=32.06, P=30.974)

# Elemental ratios per C-atom: (H/C, O/C, N/C, S/C, P/C)
COMPOSITIONS = {
    "Roels extended": (1.800, 0.500, 0.200, 0.005,  0.010),
    "Upcraft":        (1.980, 0.647, 0.173, 0.000,  0.0151),
}

def formula_str(a, b, c, d, e):
    s = f"CH{a}O{b}N{c}"
    if d: s += f"S{d}"
    s += f"P{e}"
    return s

rows = []
for name, (a, b, c, d, e) in COMPOSITIONS.items():
    MW_X    = MW['C'] + a*MW['H'] + b*MW['O'] + c*MW['N'] + d*MW['S'] + e*MW['P']
    gamma_X = 4 + a - 2*b - 3*c          # S/P contributions < 0.01 mol e⁻/C-mol each
    rows.append({"Name": name, "Formula": formula_str(a, b, c, d, e),
                 "MW (g/C-mol)": round(MW_X, 3), "γ (mol e⁻/C-mol)": round(gamma_X, 3)})

print(pd.DataFrame(rows).to_string(index=False))

## 2. Stoichiometric balance

Working per **gram of glucose consumed**.

**Carbon balance** (biomass + CO₂ = glucose carbon):
$$n_{CO_2} = n_{C,\text{glc}} - n_{C,X}$$

**Degree-of-reduction (electron) balance** (glucose electrons → biomass + O₂ reduction):
$$\gamma_S \, n_{C,\text{glc}} = \gamma_X \, n_{C,X} + 4 \, n_{O_2}$$

where $\gamma_S = 4$ mol e⁻/C-mol for glucose.

**N, S, P** — straight elemental balance from biomass composition.

**Hydrogen balance** — gives H₂O produced:
$$2\,n_{H_2O} = H_{\text{glc}} + 3\,n_{NH_3} - a\,n_{C,X}$$

In [ ]:
MW_glucose = 180.156;  Y_xs = 0.40;  gamma_S = 4.0

MW_O2  = 2*MW['O'];  MW_CO2 = MW['C'] + 2*MW['O']
MW_NH4 = MW['N'] + 4*MW['H'];  MW_SO4 = MW['S'] + 4*MW['O']
MW_PO4 = MW['P'] + 4*MW['O'] + 2*MW['H'];  MW_H2O = 2*MW['H'] + MW['O']

n_glc   = 1.0 / MW_glucose
n_C_glc = 6 * n_glc

def compute_stoich(a, b, c, d, e, Y_xs=Y_xs):
    """Return stoichiometric coefficients per gram of glucose consumed."""
    MW_X    = MW['C'] + a*MW['H'] + b*MW['O'] + c*MW['N'] + d*MW['S'] + e*MW['P']
    gamma_X = 4 + a - 2*b - 3*c
    n_C_X   = Y_xs / MW_X
    n_CO2   = n_C_glc - n_C_X
    n_O2    = (gamma_S * n_C_glc - gamma_X * n_C_X) / 4
    n_NH3   = c * n_C_X
    n_SO4   = d * n_C_X
    n_PO4   = e * n_C_X
    n_H2O   = (12*n_glc + 3*n_NH3 - a*n_C_X) / 2
    return dict(MW_X=MW_X, gamma_X=gamma_X, n_C_X=n_C_X,
                n_CO2=n_CO2, n_O2=n_O2, n_NH3=n_NH3,
                n_SO4=n_SO4, n_PO4=n_PO4, n_H2O=n_H2O,
                RQ=n_CO2/n_O2)

STOICH = {name: compute_stoich(*comps) for name, comps in COMPOSITIONS.items()}

# --- side-by-side stoichiometry table (g per g glucose) ---
SPECIES = ['Glucose', 'Biomass', 'O₂', 'CO₂', 'NH₄⁺', 'SO₄²⁻', 'H₂PO₄⁻', 'H₂O']
ROLES   = ['C+energy source', 'product', 'consumed', 'evolved',
           'N source', 'S source', 'P source', 'produced']

def g_per_g(st):
    return [-1.0, Y_xs,
            st['n_O2']*MW_O2,  st['n_CO2']*MW_CO2,
            st['n_NH3']*MW_NH4, st['n_SO4']*MW_SO4,
            st['n_PO4']*MW_PO4, st['n_H2O']*MW_H2O]

tbl = pd.DataFrame({'Species': SPECIES, 'Role': ROLES})
for name, st in STOICH.items():
    tbl[f"{name} (g/g_glc)"] = [f"{v:+.5f}" for v in g_per_g(st)]

print(tbl.to_string(index=False))
print()
for name, st in STOICH.items():
    print(f"  {name:20s}  RQ = {st['RQ']:.3f},  γ_X = {st['gamma_X']:.3f},  MW_X = {st['MW_X']:.3f} g/C-mol")

### Closure check — oxygen balance

In [ ]:
print("Oxygen balance closure per gram of glucose consumed:")
print()
for name, (a, b, c, d, e) in COMPOSITIONS.items():
    st = STOICH[name]
    O_in  = 6*n_glc + 2*st['n_O2']
    O_out = b*st['n_C_X'] + 2*st['n_CO2'] + st['n_H2O']
    print(f"  {name:20s}  O_in = {O_in:.6f},  O_out = {O_out:.6f},  "
          f"imbalance = {abs(O_in-O_out):.2e}  (mol-O / g_glc)")

## 3. Media composition vs. extent of glucose consumption

Define an initial medium, then sweep glucose consumed from 0 → S₀.

In [ ]:
S0_glucose = 20.0   # g/L initial glucose
X_0        = 0.05   # g/L inoculum biomass

# Initial medium per composition: 20% excess over stoichiometric demand
MEDIUM = {}
for name, st in STOICH.items():
    MEDIUM[name] = dict(
        NH4_0 = 1.2 * st['n_NH3'] * S0_glucose,   # mol/L
        PO4_0 = 1.2 * st['n_PO4'] * S0_glucose,   # mol/L
        SO4_0 = 1.2 * st['n_SO4'] * S0_glucose,   # mol/L
    )

print(f"{'Component':<20}  {'Roels extended':>18}  {'Upcraft':>12}")
print(f"{'Glucose (g/L)':<20}  {S0_glucose:>18.1f}  {S0_glucose:>12.1f}")
for key, label, mw in [('NH4_0', 'NH₄⁺ (mmol/L)', MW_NH4),
                        ('PO4_0', 'H₂PO₄⁻ (mmol/L)', MW_PO4),
                        ('SO4_0', 'SO₄²⁻ (mmol/L)', MW_SO4)]:
    r = MEDIUM["Roels extended"][key] * 1e3
    u = MEDIUM["Upcraft"][key] * 1e3
    print(f"{label:<20}  {r:>18.2f}  {u:>12.2f}")

In [ ]:
dS = np.linspace(0, S0_glucose, 300)   # g/L glucose consumed

# Pre-compute all sweep arrays, keyed by composition name
SWEEPS = {}
for name, st in STOICH.items():
    med = MEDIUM[name]
    SWEEPS[name] = dict(
        glucose = S0_glucose - dS,
        biomass = X_0 + Y_xs * dS,
        O2_dem  = st['n_O2']  * dS * MW_O2,          # g/L cumulative
        CO2_ev  = st['n_CO2'] * dS * 1e3,             # mmol/L cumulative
        NH4     = (med['NH4_0'] - st['n_NH3']*dS)*1e3,  # mmol/L
        PO4     = (med['PO4_0'] - st['n_PO4']*dS)*1e3,  # mmol/L
        SO4     = (med['SO4_0'] - st['n_SO4']*dS)*1e3,  # mmol/L
    )

print("Sweeps computed for:", list(SWEEPS))

## 4. Plots

In [ ]:
COLORS = {"Roels extended": "C0", "Upcraft": "C1"}
LSTYLE = {"Roels extended": "-",  "Upcraft": "--"}

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.suptitle(
    f"Aerobic growth on glucose  |  $Y_{{X/S}}$ = {Y_xs} g/g  |  "
    "blue = Roels extended,  orange = Upcraft",
    fontsize=11,
)

panels = [
    ("glucose", "Glucose",              "g / L"),
    ("biomass", "Biomass",              "g / L"),
    ("O2_dem",  "Cumulative O₂ demand", "g / L"),
    ("CO2_ev",  "Cumulative CO₂",       "mmol / L"),
    ("NH4",     "NH₄⁺",                 "mmol / L"),
    ("PO4",     "H₂PO₄⁻",              "mmol / L"),
]

for ax, (key, title, ylabel) in zip(axes.flat, panels):
    for name, sw in SWEEPS.items():
        ax.plot(dS, sw[key], color=COLORS[name], ls=LSTYLE[name], label=name)
    ax.set_title(title);  ax.set_ylabel(ylabel)
    ax.set_xlabel("Glucose consumed  (g / L)");  ax.grid(alpha=0.3)

axes.flat[2].legend(fontsize=8)   # legend on O2 panel (lines are most separated there)
plt.tight_layout()
plt.show()

## 5. Summary table at full glucose consumption

In [ ]:
rows = []
for name, st in STOICH.items():
    med = MEDIUM[name]
    rows.append({
        "Composition"      : name,
        "Glucose initial"  : f"{S0_glucose:.1f}",
        "Glucose final"    : "0",
        "Biomass initial"  : f"{X_0:.2f}",
        "Biomass final"    : f"{X_0 + Y_xs*S0_glucose:.2f}",
        "O₂ demand (g/L)"  : f"{st['n_O2']*S0_glucose*MW_O2:.2f}",
        "CO₂ evolved (g/L)": f"{st['n_CO2']*S0_glucose*MW_CO2:.2f}",
        "NH₄⁺ init (g/L)"  : f"{med['NH4_0']*MW_NH4:.2f}",
        "NH₄⁺ final (g/L)" : f"{(med['NH4_0']-st['n_NH3']*S0_glucose)*MW_NH4:.2f}",
        "H₂PO₄⁻ init (g/L)": f"{med['PO4_0']*MW_PO4:.3f}",
        "H₂PO₄⁻ final(g/L)": f"{(med['PO4_0']-st['n_PO4']*S0_glucose)*MW_PO4:.3f}",
        "RQ"               : f"{st['RQ']:.3f}",
    })

summary = pd.DataFrame(rows).T
summary.columns = list(STOICH)
print(summary.to_string())